# Lesson 7: Transformer End to End

## Overview

Earlier lessons introduced the training loop, GPU-optimized attention, and multi-GPU scaling. This lesson puts all three together: you will define a small transformer language model using **Flax NNX** modules, train it on Shakespeare text using **multi-GPU data parallelism**, save and restore checkpoints with **Orbax**, and generate new text from the trained model.

The model is intentionally tiny — 4 layers, 256-dimensional embeddings, byte-level vocabulary — so that it trains in under a minute on two L4 GPUs. The architecture and training patterns are the same ones used in much larger models.

**What you'll do:**

* Define a decoder transformer using Flax NNX: `nnx.Embed`, `nnx.MultiHeadAttention`, `nnx.Linear`, `nnx.LayerNorm`
* Plug in causal `jax.nn.dot_product_attention` as the attention backend
* Train on byte-level TinyShakespeare — no tokenizer dependency
* Use `nnx.Optimizer` with Optax AdamW for the training step
* Shard data across GPUs with the `Mesh` / `NamedSharding` pattern from Lesson 6
* Save and restore model checkpoints with Orbax `StandardCheckpointer`
* Measure throughput in **tokens/sec** on 1 GPU vs all GPUs
* Generate Shakespeare-like text from the trained model

## Decoder transformer architecture

The model is a stack of identical transformer blocks. Each block has two sub-layers with residual connections:

1. **Self-attention** — each position attends to all earlier positions (causal mask).
2. **Feed-forward network (FFN)** — two linear layers with a GELU activation, expanding and then compressing the representation.

Both sub-layers use **pre-norm**: LayerNorm is applied before the sub-layer, not after. Pre-norm is more stable for training and is the standard in modern transformers.

| Component | What it does | Lesson connection |
| --- | --- | --- |
| Token embedding | Maps each byte (0–255) to a learned vector | New — replaces L4's raw pixel input |
| Position embedding | Adds a learned vector for each sequence position | New — transformers need position info |
| Transformer block × 4 | Self-attention + FFN with residual connections | Attention backend, training from L4 |
| LM head | Projects back to vocabulary logits | Same idea as L4's output layer |
| Data parallelism | Shard batches across GPUs, replicate model | Directly from L6 |

## Requirements

The fixed NGC image and pinned workshop requirements provide:

- `jax`, `jaxlib` — core JAX with multi-GPU support
- `flax` — NNX modules for the transformer architecture
- `optax` — AdamW optimizer
- `orbax-checkpoint` — model checkpointing
- `matplotlib` — training curves

This lesson requires **at least 2 GPUs** for the multi-GPU sections. The single-GPU sections run on any GPU.

## Setup

Import JAX, Flax NNX, Optax, and Orbax. Verify that multiple GPUs are visible.

In [ ]:
import os

os.environ["LD_LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + os.environ.get(
    "LD_LIBRARY_PATH", ""
)
import hashlib
import html
import pathlib
import time
import urllib.request
import warnings

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*ml_dtypes.*")
warnings.filterwarnings("ignore", message=".*JAX_PLATFORMS.*")
import jax
import jax.numpy as jnp
import optax
import orbax.checkpoint as ocp
from flax import nnx
from jax.sharding import Mesh, NamedSharding
from jax.sharding import PartitionSpec as P

devices = jax.devices()
gpu_devices = [d for d in devices if d.platform == "gpu"]
NUM_DEVICES = len(gpu_devices)
print(f"JAX version:     {jax.__version__}")
print(f"Default backend: {jax.default_backend()}")
print(f"GPU devices:     {gpu_devices}")
print(f"GPU count:       {NUM_DEVICES}")
assert len(gpu_devices) >= 2, (
    f"This lesson needs at least 2 GPUs. Found {len(gpu_devices)}. "
    f"Available devices: {devices}"
)

def block_tree(tree):
    return jax.block_until_ready(tree)

def show_table(headers, rows, title=None, aligns=None):
    aligns = aligns or ["left"] * len(headers)
    parts = ["<div style='font-family: system-ui; max-width: 980px;'>"]
    if title:
        parts.append(f"<h4 style='margin: 0 0 8px 0;'>{html.escape(title)}</h4>")
    parts.append(
        "<table style='border-collapse: collapse; width: 100%; font-size: 13px;'>"
    )
    parts.append("<thead><tr>")
    for h, a in zip(headers, aligns):
        parts.append(
            f"<th style='text-align:{a}; border-bottom:1px solid #d0d7de; padding:6px;'>"
            f"{html.escape(str(h))}</th>"
        )
    parts.append("</tr></thead><tbody>")
    for row in rows:
        parts.append("<tr>")
        for cell, a in zip(row, aligns):
            parts.append(
                f"<td style='text-align:{a}; border-bottom:1px solid #eef1f4; padding:6px;'>"
                f"{html.escape(str(cell))}</td>"
            )
        parts.append("</tr>")
    parts.append("</tbody></table></div>")
    display(HTML("".join(parts)))

def show_bars(rows, title, unit="", lower_is_better=False):
    max_value = max(float(value) for _, value in rows) or 1.0
    color = "#1a7f37" if not lower_is_better else "#0969da"
    parts = ["<div style='font-family: Arial, sans-serif; max-width: 760px;'>"]
    parts.append(f"<h4 style='margin: 0 0 8px 0;'>{html.escape(title)}</h4>")
    for label, value in rows:
        width = max(3, 100 * float(value) / max_value)
        parts.append(
            "<div style='display:grid; grid-template-columns: 190px 1fr 130px; gap: 8px; "
            "align-items:center; margin: 6px 0;'>"
            f"<div style='font-size:13px;'>{html.escape(str(label))}</div>"
            "<div style='background:#f6f8fa; border-radius:6px; overflow:hidden; height:22px;'>"
            f"<div style='height:22px; width:{width:.1f}%; background:{color};'></div></div>"
            f"<div style='font-size:13px; font-variant-numeric: tabular-nums;'>{float(value):,.0f} {html.escape(unit)}</div>"
            "</div>"
        )
    parts.append(
        f"<div style='font-size:12px; color:#57606a;'>"
        f"{'Lower' if lower_is_better else 'Higher'} is better.</div></div>"
    )
    display(HTML("".join(parts)))


## Prepare training data

[TinyShakespeare](https://huggingface.co/datasets/karpathy/tiny_shakespeare) is a single text file (~1 MB) containing all of Shakespeare's works concatenated together. We use **byte-level tokenization**: each byte in the UTF-8 text becomes one token. This gives a fixed vocabulary of 256 possible values with zero tokenizer dependencies.

The text is chunked into non-overlapping sequences of length `SEQ_LEN`. Each sequence is a training example: the model learns to predict the next byte at every position.

In [ ]:
SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SHAKESPEARE_MD5 = "d015dc5942f9b2908e24d4827a3e7a5e"
DATA_DIR = pathlib.Path.home() / ".cache" / "jax-course"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_FILE = DATA_DIR / "tinyshakespeare.txt"

def md5sum(path):
    digest = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if DATA_FILE.exists() and md5sum(DATA_FILE) == SHAKESPEARE_MD5:
    print("Using cached tinyshakespeare.txt")
else:
    print("Downloading tinyshakespeare.txt")
    urllib.request.urlretrieve(SHAKESPEARE_URL, DATA_FILE)
raw_text = DATA_FILE.read_text()
data = np.frombuffer(raw_text.encode("utf-8"), dtype=np.uint8).astype(np.int32)
print()
VOCAB_SIZE = 256
SEQ_LEN = 256
PER_DEVICE_BATCH = 32
num_sequences = len(data) // SEQ_LEN
data = data[: num_sequences * SEQ_LEN].reshape(num_sequences, SEQ_LEN)
num_train = int(0.9 * num_sequences)
train_data = data[:num_train]
val_data = data[num_train:]
rng = np.random.default_rng(0)
train_data = train_data[rng.permutation(num_train)]

def make_batches(data, batch_size):
    usable = (len(data) // batch_size) * batch_size
    return data[:usable].reshape(-1, batch_size, SEQ_LEN)

show_table(
    ["", "Value"],
    [
        ("Total bytes", f"{len(raw_text):,}"),
        ("Vocabulary", f"{VOCAB_SIZE} (raw bytes)"),
        ("Sequence length", SEQ_LEN),
        ("Training sequences", f"{num_train:,}"),
        ("Validation sequences", f"{len(val_data):,}"),
    ],
    title="TinyShakespeare — byte-level tokenization",
)
print()
print("Sample text (first 200 bytes):")
print(raw_text[:200])

## Define the transformer with Flax NNX

[Flax NNX](https://flax.readthedocs.io/en/latest/nnx_basics.html) is a module system for JAX. Instead of writing raw matrix multiplies like Lesson 4, you define layers as Python objects that handle weight initialization and the forward pass. This lesson introduces the NNX pieces needed for a full decoder transformer, so no earlier NNX preview is assumed.

The key NNX building blocks used here:

| NNX layer | What it does | L4 equivalent |
| --- | --- | --- |
| `nnx.Embed` | Lookup table: token index -> vector | Manual `params["embed"][tokens]` |
| `nnx.Linear` | Dense matrix multiply + optional bias | Manual `x @ params["w"] + params["b"]` |
| `nnx.LayerNorm` | Normalize features before attention/FFN blocks | Not used in L4 |
| `nnx.MultiHeadAttention` | Q/K/V projections + attention + output projection | Combines learned projections with SDPA |

`nnx.MultiHeadAttention` receives hidden states shaped `(B, T, D_MODEL)`, then creates Q, K, and V internally and splits them into heads. The `attention_fn` hook controls only the core attention operation after those projections. NNX passes Flax-style optional arguments such as dropout, dtype, and precision into `attention_fn`, so our small `causal_sdpa(..., **_)` wrapper accepts those extra arguments and forwards only what `jax.nn.dot_product_attention` needs.

The model has two classes: `TransformerBlock` (one attention + FFN sub-layer) and `TinyTransformer` (the full model stacking N blocks). Each class extends `nnx.Module` and creates all its layers in `__init__`.


In [ ]:
D_MODEL = 256
NUM_HEADS = 4
FFN_DIM = 1024
NUM_LAYERS = 4
MAX_SEQ_LEN = 256
LR = 3e-4
WEIGHT_DECAY = 1e-4

# NNX passes extra attention kwargs such as dropout_rng, dtype, and precision.
# This wrapper keeps the signature compatible while selecting causal JAX SDPA.
def causal_sdpa(query, key, value, **_):
    return jax.nn.dot_product_attention(query, key, value, is_causal=True)

class TransformerBlock(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, ffn_dim: int, rngs: nnx.Rngs):
        self.ln1 = nnx.LayerNorm(d_model, rngs=rngs)
        self.attn = nnx.MultiHeadAttention(
            num_heads=num_heads,
            in_features=d_model,
            decode=False,
            attention_fn=causal_sdpa,
            rngs=rngs,
        )
        self.ln2 = nnx.LayerNorm(d_model, rngs=rngs)
        self.fc_up = nnx.Linear(d_model, ffn_dim, rngs=rngs)
        self.fc_down = nnx.Linear(ffn_dim, d_model, rngs=rngs)

    def __call__(self, x):
        x = x + self.attn(self.ln1(x))
        h = jax.nn.gelu(self.fc_up(self.ln2(x)))
        x = x + self.fc_down(h)
        return x

class TinyTransformer(nnx.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        num_heads: int,
        ffn_dim: int,
        num_layers: int,
        max_seq_len: int,
        rngs: nnx.Rngs,
    ):
        self.token_embed = nnx.Embed(vocab_size, d_model, rngs=rngs)
        self.pos_embed = nnx.Embed(max_seq_len, d_model, rngs=rngs)
        self.blocks = nnx.List(
            [
                TransformerBlock(d_model, num_heads, ffn_dim, rngs=rngs)
                for _ in range(num_layers)
            ]
        )
        self.final_norm = nnx.LayerNorm(d_model, rngs=rngs)
        self.lm_head = nnx.Linear(d_model, vocab_size, use_bias=False, rngs=rngs)

    def __call__(self, tokens):
        _, T = tokens.shape
        x = self.token_embed(tokens) + self.pos_embed(jnp.arange(T))
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        return self.lm_head(x)

## Instantiate and inspect the model

Creating the model is one line — `nnx.Rngs` handles all random state for parameter initialization. After creation, we can inspect the parameter shapes and count.

In [ ]:
model = TinyTransformer(
    VOCAB_SIZE,
    D_MODEL,
    NUM_HEADS,
    FFN_DIM,
    NUM_LAYERS,
    MAX_SEQ_LEN,
    rngs=nnx.Rngs(0),
)
param_count = sum(x.size for x in jax.tree.leaves(nnx.state(model, nnx.Param)))
show_table(
    ["", "Value"],
    [
        ("Architecture", "Decoder-only transformer"),
        ("Layers", NUM_LAYERS),
        ("Model dimension", D_MODEL),
        ("Attention heads", f"{NUM_HEADS} (head dim = {D_MODEL // NUM_HEADS})"),
        ("FFN dimension", FFN_DIM),
        ("Vocabulary", f"{VOCAB_SIZE} (byte-level)"),
        ("Max sequence length", MAX_SEQ_LEN),
        ("Parameters", f"{param_count:,}"),
    ],
    title="TinyTransformer",
)
logits = model(jnp.zeros((1, 16), dtype=jnp.int32))
print(f"Test forward pass: input (1, 16) \u2192 logits {logits.shape}")

## The training step

The training step follows the same pattern as Lesson 4, now using NNX modules:

| L4 (raw PyTree) | L7 (Flax NNX) |
| --- | --- |
| `jax.value_and_grad(loss)(params)` | `nnx.value_and_grad(loss)(model)` |
| `optax.update(grads, opt_state, params)` | `optimizer.update(model, grads)` |
| `@jax.jit` | `@nnx.jit` |

`@nnx.jit` handles NNX module state automatically — it splits modules into structure + arrays for JIT compilation, then merges the updated arrays back. You write the step as if modules were regular Python objects.

For the loss: at each position, the model predicts the next token. We compare `logits[:, :-1]` (predictions at positions 0 to T-2) with `tokens[:, 1:]` (actual tokens at positions 1 to T-1).

In [ ]:
@nnx.jit
def train_step(model, optimizer, tokens):
    def loss_fn(model):
        logits = model(tokens)
        pred = logits[:, :-1].reshape(-1, VOCAB_SIZE)
        target = tokens[:, 1:].reshape(-1)
        return optax.softmax_cross_entropy_with_integer_labels(pred, target).mean()

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return {"loss": loss, "perplexity": jnp.exp(loss)}

def train_loop(model, optimizer, batches, steps=1000, log_every=100):
    num_batches = batches.shape[0]
    history = []
    # Warmup: compile the training step
    warmup_metrics = train_step(model, optimizer, batches[0])
    block_tree(warmup_metrics)
    start = time.perf_counter()
    for step in range(steps):
        tokens = batches[step % num_batches]
        metrics = train_step(model, optimizer, tokens)
        if step % log_every == 0 or step == steps - 1:
            metrics = block_tree(metrics)
            history.append(
                {
                    "step": step,
                    "loss": float(metrics["loss"]),
                    "perplexity": float(metrics["perplexity"]),
                }
            )
    block_tree(metrics)
    elapsed = time.perf_counter() - start
    batch_size = int(batches.shape[1])
    tokens_per_step = batch_size * (SEQ_LEN - 1)
    tokens_per_sec = steps * tokens_per_step / elapsed
    return history, elapsed, tokens_per_sec

## Train on a single GPU

First, train on one GPU to establish a baseline. This is the same idea as Lesson 4's training loop, but with a transformer instead of an MLP and text instead of images.

In [ ]:
BENCHMARK_STEPS = 500
STEPS_1GPU = BENCHMARK_STEPS
batches_1gpu = make_batches(train_data, PER_DEVICE_BATCH)
single_device = gpu_devices[0]
batches_1gpu = jax.device_put(batches_1gpu, single_device)
model_1gpu = TinyTransformer(
    VOCAB_SIZE,
    D_MODEL,
    NUM_HEADS,
    FFN_DIM,
    NUM_LAYERS,
    MAX_SEQ_LEN,
    rngs=nnx.Rngs(1),
)
optimizer_1gpu = nnx.Optimizer(
    model_1gpu, optax.adamw(LR, weight_decay=WEIGHT_DECAY), wrt=nnx.Param
)
history_1gpu, elapsed_1gpu, tps_1gpu = train_loop(
    model_1gpu, optimizer_1gpu, batches_1gpu, steps=STEPS_1GPU
)
show_table(
    ["Step", "Loss", "Perplexity"],
    [(h["step"], f"{h['loss']:.3f}", f"{h['perplexity']:.1f}") for h in history_1gpu],
    title=f"Single-GPU training \u2014 {tps_1gpu:,.0f} tokens/sec",
    aligns=["right", "right", "right"],
)

## Scale to multiple GPUs

This is the same data-parallel pattern from Lesson 6: create a mesh, replicate the model, shard the data along the batch dimension. The training step code does not change — `@nnx.jit` handles the parallelism automatically based on how the arrays are placed.

To keep the throughput comparison fair, the single-GPU and multi-GPU runs use the same number of timed steps. Each GPU still processes `PER_DEVICE_BATCH` sequences per step, while the multi-GPU run processes a larger global batch.

To replicate the model across GPUs, we extract its state with `nnx.state`, place it on all devices with `jax.device_put`, and write it back with `nnx.update`. The same goes for the optimizer.

In [ ]:
STEPS_MULTI = BENCHMARK_STEPS
GLOBAL_BATCH = PER_DEVICE_BATCH * NUM_DEVICES
mesh = Mesh(np.array(gpu_devices), ("data",))
replicated = NamedSharding(mesh, P())
data_sharding = NamedSharding(mesh, P(None, "data", None))
batches_multi = make_batches(train_data, GLOBAL_BATCH)
batches_multi = jax.device_put(batches_multi, data_sharding)
model_multi = TinyTransformer(
    VOCAB_SIZE,
    D_MODEL,
    NUM_HEADS,
    FFN_DIM,
    NUM_LAYERS,
    MAX_SEQ_LEN,
    rngs=nnx.Rngs(1),
)
optimizer_multi = nnx.Optimizer(
    model_multi, optax.adamw(LR, weight_decay=WEIGHT_DECAY), wrt=nnx.Param
)
# Replicate model and optimizer state across all GPUs
model_state = nnx.state(model_multi)
nnx.update(model_multi, jax.device_put(model_state, replicated))
opt_state = nnx.state(optimizer_multi)
nnx.update(optimizer_multi, jax.device_put(opt_state, replicated))
print(
    f"Global batch: {GLOBAL_BATCH} ({PER_DEVICE_BATCH} per GPU \u00d7 {NUM_DEVICES} GPUs)"
)
print(f"Training batches: {batches_multi.shape}")
print()
history_multi, elapsed_multi, tps_multi = train_loop(
    model_multi, optimizer_multi, batches_multi, steps=STEPS_MULTI
)
show_table(
    ["Step", "Loss", "Perplexity"],
    [(h["step"], f"{h['loss']:.3f}", f"{h['perplexity']:.1f}") for h in history_multi],
    title=f"Multi-GPU training \u2014 {tps_multi:,.0f} tokens/sec",
    aligns=["right", "right", "right"],
)

## Single-GPU vs multi-GPU throughput

The model and training step are identical. Only the data placement changed. Both runs use the same number of timed steps, and each GPU still processes `PER_DEVICE_BATCH` sequences per step. The multi-GPU run has a larger global batch, so this is a weak-scaling throughput comparison in tokens/sec. Use ms/step as a sanity check alongside tokens/sec.

In [ ]:
ms_per_step_1gpu = elapsed_1gpu / STEPS_1GPU * 1e3
ms_per_step_multi = elapsed_multi / STEPS_MULTI * 1e3
speedup = tps_multi / tps_1gpu

show_table(
    ["", "1 GPU", f"{NUM_DEVICES} GPUs", "Ratio"],
    [
        ("Batch size", PER_DEVICE_BATCH, GLOBAL_BATCH, f"{NUM_DEVICES}×"),
        ("Per-GPU batch", PER_DEVICE_BATCH, PER_DEVICE_BATCH, "same"),
        ("Timed steps", STEPS_1GPU, STEPS_MULTI, "same"),
        ("ms/step", f"{ms_per_step_1gpu:.2f}", f"{ms_per_step_multi:.2f}", f"{ms_per_step_1gpu / ms_per_step_multi:.2f}×"),
        ("Tokens/sec", f"{tps_1gpu:,.0f}", f"{tps_multi:,.0f}", f"{speedup:.2f}×"),
    ],
    title="Throughput comparison — same timed steps",
    aligns=["left", "right", "right", "right"],
)

if speedup > NUM_DEVICES * 1.25:
    print(
        f"Note: the measured speedup is superlinear (> {NUM_DEVICES}x). "
        "For this small benchmark, treat that as a measurement artifact rather "
        "than a general hardware-scaling claim."
    )

show_bars(
    [("1 GPU", tps_1gpu), (f"{NUM_DEVICES} GPUs", tps_multi)],
    "Training throughput",
    "tokens/s",
)

The 2-GPU run is faster because it processes a larger global batch while keeping the per-GPU batch fixed. If the measured speedup is greater than the number of GPUs, treat it as a benchmark artifact from compiler/layout/kernel differences rather than a general scaling guarantee.

This code below visualizes a plot comparing training progress for the single-GPU and multi-GPU runs as the model sees more tokens. The x-axis is tokens processed rather than raw training steps, because the multi-GPU run uses a larger global batch and therefore sees more data per step. Lower loss and perplexity are better, so the curves show how quickly each setup improves for the amount of text processed.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for label, history, batch_size in [
    ("1 GPU", history_1gpu, PER_DEVICE_BATCH),
    (f"{NUM_DEVICES} GPUs", history_multi, GLOBAL_BATCH),
]:
    tokens_m = [
        (h["step"] + 1) * batch_size * (SEQ_LEN - 1) / 1e6 for h in history
    ]
    losses = [h["loss"] for h in history]
    perps = [h["perplexity"] for h in history]
    ax1.plot(tokens_m, losses, "o-", label=label, markersize=4)
    ax2.plot(tokens_m, perps, "o-", label=label, markersize=4)
ax1.set_xlabel("Tokens processed (millions)")
ax1.set_ylabel("Loss")
ax1.set_title("Training loss")
ax1.legend()
ax1.grid(True, alpha=0.25)
ax2.set_xlabel("Tokens processed (millions)")
ax2.set_ylabel("Perplexity")
ax2.set_title("Training perplexity")
ax2.legend()
ax2.grid(True, alpha=0.25)
fig.suptitle("Training curves vs tokens processed")
fig.tight_layout()
plt.show()

## Save and restore a checkpoint with Orbax

[Orbax](https://orbax.readthedocs.io) saves model state as a directory of array files. `StandardCheckpointer` is the simplest API: one call to save, one call to restore.

For NNX models, we extract parameters with `nnx.state(model, nnx.Param)`, save them, and later restore into a fresh model with `nnx.update`. The graph structure (`nnx.GraphDef`) is not saved — it comes from the Python class definition.

This cell demonstrates the checkpoint lifecycle with Orbax: it extracts only the trained model parameters, saves them to disk, restores them into a newly initialized `TinyTransformer`, and verifies that the restored model produces the same logits as the original model on a small test input. The `ShapeDtypeStruct` tree tells Orbax what parameter shapes and dtypes to expect during restore, while the final max-difference check confirms that checkpointing preserved the learned weights correctly.

In [ ]:
ckpt_dir = pathlib.Path("/tmp/jax-course/l7-checkpoints")
# Extract model parameters (not optimizer state)
model_params = nnx.state(model_multi, nnx.Param)
# Save
checkpointer = ocp.StandardCheckpointer()
if (ckpt_dir / "trained").exists():
    import shutil

    shutil.rmtree(ckpt_dir / "trained")
checkpointer.save(ckpt_dir / "trained", model_params)
print(f"Checkpoint saved to {ckpt_dir / 'trained'}")
# Create abstract target for restore (same structure, ShapeDtypeStruct leaves)
abstract_params = jax.tree.map(
    lambda x: jax.ShapeDtypeStruct(x.shape, x.dtype),
    model_params,
)
# Restore into a fresh model
model_restored = TinyTransformer(
    VOCAB_SIZE,
    D_MODEL,
    NUM_HEADS,
    FFN_DIM,
    NUM_LAYERS,
    MAX_SEQ_LEN,
    rngs=nnx.Rngs(99),
)
restored_params = checkpointer.restore(ckpt_dir / "trained", abstract_params)
nnx.update(model_restored, restored_params)
# Verify: same output on a test input
test_input = jnp.zeros((1, 16), dtype=jnp.int32)
logits_original = model_multi(test_input)
logits_restored = model_restored(test_input)
max_diff = float(jnp.max(jnp.abs(logits_original - logits_restored)))
show_table(
    ["", "Value"],
    [
        ("Checkpoint path", str(ckpt_dir / "trained")),
        ("Parameters saved", f"{sum(x.size for x in jax.tree.leaves(model_params)):,}"),
        ("Max |original \u2212 restored|", f"{max_diff:.2e}"),
        ("Match", "\u2713" if max_diff < 1e-5 else "\u2717"),
    ],
    title="Orbax checkpoint save and restore",
)

## Generate text

The trained model predicts the next byte at every position. To generate text, we feed a prompt, take the logits at the last position, sample a token, append it, and repeat.

The `generate` function pads input to `MAX_SEQ_LEN` so the JIT-compiled forward pass always sees the same input shape — no recompilation as the sequence grows. With causal attention, padding after the real tokens does not affect the output at earlier positions.

In [ ]:
@jax.jit
def get_logits_jit(graphdef, model_state, tokens):
    model = nnx.merge(graphdef, model_state)
    return model(tokens)

def generate(model, prompt_text, max_new_tokens=300, temperature=0.8):
    graphdef, model_state = nnx.split(model)
    tokens = list(prompt_text.encode("utf-8"))
    key = jax.random.key(42)
    for _ in range(max_new_tokens):
        context = tokens[-MAX_SEQ_LEN:]
        padded = context + [0] * (MAX_SEQ_LEN - len(context))
        input_arr = jnp.array([padded], dtype=jnp.int32)
        logits = get_logits_jit(graphdef, model_state, input_arr)
        next_logit = logits[0, len(context) - 1]
        if temperature <= 0:
            next_token = int(jnp.argmax(next_logit))
        else:
            key, subkey = jax.random.split(key)
            next_token = int(jax.random.categorical(subkey, next_logit / temperature))
        tokens.append(next_token)
    return bytes(tokens).decode("utf-8", errors="replace")

# Put model on a single device for generation
gen_model = TinyTransformer(
    VOCAB_SIZE,
    D_MODEL,
    NUM_HEADS,
    FFN_DIM,
    NUM_LAYERS,
    MAX_SEQ_LEN,
    rngs=nnx.Rngs(99),
)
nnx.update(gen_model, checkpointer.restore(ckpt_dir / "trained", abstract_params))
print("=== Prompt: 'ROMEO:' | temperature=0.8 ===")
print()
print(generate(gen_model, "ROMEO:", max_new_tokens=300, temperature=0.8))
print()
print("=== Prompt: 'To be, or not' | temperature=0.6 ===")
print()
print(generate(gen_model, "To be, or not", max_new_tokens=300, temperature=0.6))

## Try changing these knobs

| Knob | What to change | What to watch |
| ---- | -------------- | ------------- |
| Model dimension | `D_MODEL` (try 128, 512) | Larger = more capacity but slower |
| Number of layers | `NUM_LAYERS` (try 2, 6) | More layers = more capacity, more memory |
| Sequence length | `SEQ_LEN` (try 128, 512) | Longer context = better generation, more memory |
| Learning rate | `LR` (try 1e-4, 1e-3) | Too high = unstable, too low = slow progress |
| Temperature | In `generate()` (try 0.0, 0.5, 1.2) | 0 = greedy, >1 = more random |
| Attention backend | Change the body of `causal_sdpa` to call `jax.nn.dot_product_attention(..., is_causal=True, implementation="cudnn")`, and use bf16-compatible activations | cuDNN fused attention behavior |
| Training steps | `BENCHMARK_STEPS` | Keep the timed step count equal for throughput comparisons; increase it for longer training |

Keep the wrapper form for attention backend experiments: passing `jax.nn.dot_product_attention` directly to `nnx.MultiHeadAttention` will not accept the extra Flax-style kwargs that NNX supplies.


## Summary

This lesson combined the training loop from Lesson 4, the attention with Flax NNX, and the multi-GPU sharding from Lesson 6 into a complete transformer language model.

* **Flax NNX** organizes the model into reusable modules. `nnx.Embed`, `nnx.Linear`, `nnx.LayerNorm`, and `nnx.MultiHeadAttention` handle parameter creation and the forward pass. The training step uses `nnx.value_and_grad` and `nnx.Optimizer`.
* **Causal attention** with `is_causal=True` masks future positions so the model can only look backward.
* **Data-parallel training** replicates the model and shards the batch across GPUs, exactly as in Lesson 6. The training step code does not change.
* **Orbax** saves and restores model parameters. `StandardCheckpointer` handles the serialization; NNX's `nnx.state` and `nnx.update` bridge between modules and plain PyTrees.
* **Throughput** is measured in tokens/sec — the natural unit for language models. Warmup and `block_until_ready` are still essential for accurate timing.
* **Text generation** feeds the model its own predictions one token at a time. Temperature controls randomness: 0 is greedy, higher values produce more diverse output.

In the next lesson, you will take this trained model and prepare it for serving: JIT inference, AOT compilation, and exporting to portable formats.

Official references:

* [Flax NNX basics](https://flax.readthedocs.io/en/latest/nnx/nnx_basics.html)
* [Flax NNX and JAX transforms](https://flax.readthedocs.io/en/latest/guides/jax_and_nnx_transforms.html)
* [Orbax checkpoint guide](https://orbax.readthedocs.io/en/latest/guides/checkpoint/orbax_checkpoint_101.html)
* [nnx.MultiHeadAttention API](https://flax.readthedocs.io/en/latest/api_reference/flax.nnx/nn/attention.html)
* [Optax AdamW](https://optax.readthedocs.io/en/stable/api/optimizers.html#adamw)